# Aula 07

## Threads

Da [Wikipedia](https://pt.wikipedia.org/wiki/Thread_(computa%C3%A7%C3%A3o)):

> Uma ***thread*** é a menor sequência de instruções programadas que pode ser gerenciada independentemente por um agendador (*scheduler*), que normalmente faz parte do sistema operacional. Em muitos casos, uma ***thread*** é um componente de um processo.
>
> As múltiplas ***threads*** de um determinado processo podem ser executadas concorrentemente (através de recursos de *multithreading*), compartilhando recursos como **memória**, enquanto processos diferentes não compartilham esses recursos. Em particular, as ***threads*** de um processo **compartilham** seu **código executável** e os **valores de suas variáveis ​​alocadas dinamicamente** e **variáveis ​​globais não locais da *thread*** em qualquer momento.
>
> O suporte à *thread* é fornecido pelo sistema operacional no caso da linha de execução ao nível do núcleo (em inglês: *Kernel-Level Thread*, KLT), ou implementada através de uma biblioteca de uma determinada linguagem de programação (*User-Level Thread*, ULT). Uma thread permite, por exemplo, que o utilizador de um programa utilize uma funcionalidade do ambiente enquanto outras linhas de execução realizam outros cálculos e operações.
>
> Os sistemas que suportam uma única *thread* (em real execução) são chamados de ***single-thread*** enquanto que os **sistemas que suportam múltiplas *threads*** são chamados de ***multithread*** (multitarefa).

<div style="display:flex; justify-content: center; align-items: center; gap: 10px;">
<figure style="max-width: 100%; height: auto;">
    <img src="./imagens/Klt.jpg">
    <figcaption><i>Thread</i> em modo kernel.</figcaption>
</figure>

<figure style="max-width: 100%; height: auto;">
    <img src="./imagens/Ult.jpg">
    <figcaption><i>Thread</i> em modo usuário.</figcaption>
</figure>
</div>

## O problema da Interface Gráfica travada

Aplicações baseadas em Qt  são baseadas em **eventos**. Isso significa que a execução é orientada a eventos em resposta à interação do usuário, sinais e temporizadores. Em uma aplicação orientada a eventos, clicar em um botão cria um evento que a aplicação processa para produzir a saída esperada. Os **eventos são adicionados e removidos de uma fila de eventos e processados ​​sequencialmente**.

No PySide6 criamos uma aplicação com o seguinte código

```python
app = QApplication([])
window = MainWindow()
app.exec()
```

O loop de eventos (*event loop*) inicia quando se chama o método `.exec()` no objeto `QApplication`, e é executado na mesma ***thread*** do código Python. A ***thread*** que executa esse loop de eventos — geralmente chamada de ***thread*** da GUI — também lida com toda a comunicação da janela com o sistema operacional.

Por padrão, qualquer execução acionada pelo loop de eventos também será executada de forma **síncrona** dentro desta ***thread***. Na prática, isso significa que durante o tempo em que o aplicativo PySide6 fica realizando alguma tarefa, a comunicação com a janela e a interação com a interface gráfica ficam congeladas.

Se o programa for simples e retornar o controle ao loop da interface gráfica rapidamente, o congelamento da interface será imperceptível para o usuário. No entanto, se for necessário executar tarefas mais demoradas, como abrir e gravar um arquivo grande, baixar dados ou renderizar uma imagem de alta resolução, haverá problemas.

Para o usuário, o aplicativo parecerá não responder (porque não vai responder mesmo). Ninguém quer isso. A solução é mover as tarefas de longa duração da ***thread*** da GUI para outra ***thread***, e o PySide6 fornece uma interface simples para isso.

### Exemplo da GUI sendo travada

In [4]:
import time

from PySide6.QtCore import QTimer
from PySide6.QtWidgets import (
    QApplication,
    QLabel,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QWidget,
)

class MainWindow(QMainWindow):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.counter = 0

        layout = QVBoxLayout()

        self.label = QLabel("Início")
        button = QPushButton("PRE-RI-GO!")
        button.pressed.connect(self.oh_no)

        layout.addWidget(self.label)
        layout.addWidget(button)

        w = QWidget()
        w.setLayout(layout)
        self.setCentralWidget(w)

        # Outra possibilidade de fazer a Janela Principal ser mostrada
        self.show()

        self.timer = QTimer()
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.recurring_timer)
        self.timer.start()

    def oh_no(self):
        time.sleep(5)

    def recurring_timer(self):
        self.counter += 1
        self.label.setText(f"Counter: {self.counter}")


# Por causa do notebook, precisamos ver se já existe uma instância do QApplication, caso contrário, criamos uma nova.
if not QApplication.instance():
    app = QApplication([])
else:
    app = QApplication.instance()

window = MainWindow()
# Aqui não tem window.show() por causa da linha 32
app.exec()

0

O que parece ser uma interface travada é o loop de eventos sendo bloqueado de processar e responder a eventos de janela. Os eventos ainda são registrados pelo Sistema Operacional e enviados à aplicação, mas por causa do evento "bloqueante", a aplicação não pode aceitar ou reagir aos demais eventos.

## Usando ***threads*** no PySide6

O Qt fornece uma interface simples para executar tarefas ou trabalhos em outras ***threads***, que é bem suportada no PySide6. Essa interface é construída em torno de duas classes:

- [`QRunnable`](https://doc.qt.io/qtforpython-6/PySide6/QtCore/QRunnable.html): o container para cada tarefa (*work*).
- [`QThreadPool`](https://doc.qt.io/qtforpython-6/PySide6/QtCore/QThreadPool.html): o método pelo qual a tarefa (*work*) será passada para ***threads*** alternativas.

A grande vantagem de usar o `QThreadPool` é que ele gerencia o enfileiramento e a execução dos `workers` (os containeres das tarefas/*works*) para você. Além de enfileirar as tarefas e recuperar os resultados, não há muito mais a fazer.

O `Exemplo 01` atualizado fica assim:

In [ ]:
from PySide6.QtCore import QRunnable, QThreadPool, QTimer, Slot
from PySide6.QtWidgets import (
    QApplication,
    QLabel,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QWidget,
)

class Worker(QRunnable):
    """Thread de tarefa/trabalho."""

    @Slot()
    def run(self):
        """A tarefa de longa execução."""
        print("Thread iniciada")
        time.sleep(5)
        print("Thread completa")

class MainWindow(QMainWindow):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.counter = 0

        layout = QVBoxLayout()

        self.label = QLabel("Início")
        button = QPushButton("PRE-RI-GO!")
        button.pressed.connect(self.oh_no)

        layout.addWidget(self.label)
        layout.addWidget(button)

        w = QWidget()
        w.setLayout(layout)
        self.setCentralWidget(w)

        self.show()

        self.timer = QTimer()
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.recurring_timer)
        self.timer.start()

        self.threadpool = QThreadPool()
        thread_count = self.threadpool.maxThreadCount()
        print(f"Multithreading com o máximo de {thread_count} threads")

    def oh_no(self):
        worker = Worker()
        self.threadpool.start(worker)

    def recurring_timer(self):
        self.counter += 1
        self.label.setText(f"Counter: {self.counter}")


# Por causa do notebook, precisamos ver se já existe uma instância do QApplication, caso contrário, criamos uma nova.
if not QApplication.instance():
    app = QApplication([])
else:
    app = QApplication.instance()

window = MainWindow()
app.exec()

Multithreading com o máximo de 12 threads
Thread iniciada
Thread completa


0

### Exemplo finalizado

In [9]:
import sys
import time
import traceback

from PySide6.QtCore import (
    QObject,
    QRunnable,
    QThreadPool,
    QTimer,
    Signal,
    Slot,
)
from PySide6.QtWidgets import (
    QApplication,
    QLabel,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QWidget,
)

class WorkerSignals(QObject):
    """Sinais de uma thread de trabalho em execução.

    finished
        int thread_id

    error
        tuple (exctype, value, traceback.format_exc())

    result
        dados dos objeto retornados do processamento, qualquer tipo

    progress
        tuple (thread_id, progress_value)
    """

    finished = Signal(int)  # thread_id
    error = Signal(tuple)
    result = Signal(object)
    progress = Signal(tuple)  # (thread_id, progress_value)

class Worker(QRunnable):
    """Thread de tarefa/trabalho.

    Herda de QRunnable para lidar com a configuração, os sinais e o encerramento da thread de trabalho.

    :param callback: A função callback a ser executada nessa thread de trabalho.
                     Args e kwargs inseridos serão passados para o executor (runner).
    :type callback: function
    :param args: Argumentos a serem passados para a função callback
    :param kwargs: "palavra=chave" a serem passados para a função callback
    """

    def __init__(self, fn, *args, **kwargs):
        super().__init__()
        self.fn = fn
        self.args = args
        self.kwargs = kwargs
        self.signals = WorkerSignals()
        self.thread_id = kwargs.get("thread_id", 0)
        # Add the callback to our kwargs
        self.kwargs["progress_callback"] = self.signals.progress

    @Slot()
    def run(self):
        try:
            result = self.fn(*self.args, **self.kwargs)
        except Exception:
            traceback.print_exc()
            exctype, value = sys.exc_info()[:2]
            self.signals.error.emit((exctype, value, traceback.format_exc()))
        else:
            self.signals.result.emit(result)
        finally:
            self.signals.finished.emit(self.thread_id)

class MainWindow(QMainWindow):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.counter = 0
        self.thread_id = 0

        layout = QVBoxLayout()

        self.label = QLabel("Início")
        button = QPushButton("PRE-RI-GO!")
        button.pressed.connect(self.oh_no)

        layout.addWidget(self.label)
        layout.addWidget(button)

        w = QWidget()
        w.setLayout(layout)
        self.setCentralWidget(w)

        self.show()

        self.threadpool = QThreadPool()
        thread_count = self.threadpool.maxThreadCount()
        print(f"Multithreading com o máximo de {thread_count} threads")

        self.timer = QTimer()
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.recurring_timer)
        self.timer.start()

    def progress_fn(self, data):
        thread_id, n = data
        print(f"THREAD #{thread_id}: {n:.1f}% concluída")

    def execute_this_fn(self, progress_callback, thread_id):
        for n in range(0, 5):
            time.sleep(1)
            progress = n * 100 / 4
            progress_callback.emit((thread_id, progress))
        return "Done."

    def print_output(self, s):
        print(s)

    def thread_complete(self, thread_id):
        print(f"THREAD #{thread_id} FINALIZADA!")

    def oh_no(self):
        # Pass the function to execute
        self.thread_id += 1
        worker = Worker(
            self.execute_this_fn, thread_id=self.thread_id
        )  # Any other args, kwargs are passed to the run function
        worker.signals.result.connect(self.print_output)
        worker.signals.finished.connect(self.thread_complete)
        worker.signals.progress.connect(self.progress_fn)
        # Execute
        self.threadpool.start(worker)

    def recurring_timer(self):
        self.counter += 1
        self.label.setText(f"Contador: {self.counter}")

# Por causa do notebook, precisamos ver se já existe uma instância do QApplication, caso contrário, criamos uma nova.
if not QApplication.instance():
    app = QApplication([])
else:
    app = QApplication.instance()

window = MainWindow()
app.exec()

Multithreading com o máximo de 12 threads
THREAD #1: 0.0% concluída
THREAD #1: 25.0% concluída
THREAD #1: 50.0% concluída
THREAD #1: 75.0% concluída
THREAD #2: 0.0% concluída
THREAD #1: 100.0% concluída
Done.
THREAD #1 FINALIZADA!
THREAD #2: 25.0% concluída
THREAD #2: 50.0% concluída
THREAD #2: 75.0% concluída
THREAD #3: 0.0% concluída
THREAD #2: 100.0% concluída
Done.
THREAD #2 FINALIZADA!
THREAD #3: 25.0% concluída
THREAD #3: 50.0% concluída
THREAD #4: 0.0% concluída
THREAD #3: 75.0% concluída
THREAD #5: 0.0% concluída
THREAD #4: 25.0% concluída
THREAD #3: 100.0% concluída
Done.
THREAD #3 FINALIZADA!
THREAD #5: 25.0% concluída
THREAD #4: 50.0% concluída
THREAD #5: 50.0% concluída
THREAD #4: 75.0% concluída
THREAD #5: 75.0% concluída
THREAD #4: 100.0% concluída
Done.
THREAD #4 FINALIZADA!
THREAD #5: 100.0% concluída
Done.
THREAD #5 FINALIZADA!


0

## Usando ***threads*** nativas do Python

Agora o mesmo exemplo anterior, porém utilizado ***threads*** a partir de módulos nativos do Python, em vez das classes fornecidas pelo PySide6.

In [10]:
import sys
import time
import traceback
import threading

from PySide6.QtCore import (
    QObject,
    QTimer,
    Signal,
)
from PySide6.QtWidgets import (
    QApplication,
    QLabel,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QWidget,
)


class WorkerSignals(QObject):
    """Sinais de uma thread de trabalho em execução.

    finished
        int thread_id

    error
        tuple (exctype, value, traceback.format_exc())

    result
        dados do objeto retornados do processamento, qualquer tipo

    progress
        tuple (thread_id, progress_value)
    """

    finished = Signal(int)   # thread_id
    error = Signal(tuple)
    result = Signal(object)
    progress = Signal(tuple)  # (thread_id, progress_value)


class Worker:
    """Thread de tarefa/trabalho.

    Usa `threading.Thread` nativo do Python para executar a função callback
    em segundo plano, comunicando-se com a UI por meio de sinais do Qt.

    :param fn: A função callback a ser executada nessa thread de trabalho.
               Args e kwargs inseridos serão passados para o executor (runner).
    :type fn: function
    :param args: Argumentos a serem passados para a função callback
    :param kwargs: "palavra=chave" a serem passados para a função callback
    """

    def __init__(self, fn, *args, **kwargs):
        self.fn = fn
        self.args = args
        self.kwargs = kwargs
        self.signals = WorkerSignals()
        self.thread_id = kwargs.get("thread_id", 0)
        # Adiciona o callback de progresso aos kwargs
        self.kwargs["progress_callback"] = self.signals.progress

        # Cria a thread nativa do Python (daemon=True encerra junto com o app)
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        """Método interno executado pela thread nativa."""
        try:
            result = self.fn(*self.args, **self.kwargs)
        except Exception:
            traceback.print_exc()
            exctype, value = sys.exc_info()[:2]
            self.signals.error.emit((exctype, value, traceback.format_exc()))
        else:
            self.signals.result.emit(result)
        finally:
            self.signals.finished.emit(self.thread_id)

    def start(self):
        """Inicia a thread nativa."""
        self._thread.start()


class MainWindow(QMainWindow):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.counter = 0
        self.thread_id = 0

        layout = QVBoxLayout()

        self.label = QLabel("Início")
        button = QPushButton("PRE-RI-GO!")
        button.pressed.connect(self.oh_no)

        layout.addWidget(self.label)
        layout.addWidget(button)

        w = QWidget()
        w.setLayout(layout)
        self.setCentralWidget(w)

        self.show()

        max_threads = threading.active_count()
        print(f"Usando threading nativo do Python (threads ativas: {max_threads})")

        self.timer = QTimer()
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.recurring_timer)
        self.timer.start()

    def progress_fn(self, data):
        thread_id, n = data
        print(f"THREAD #{thread_id}: {n:.1f}% concluída")

    def execute_this_fn(self, progress_callback, thread_id):
        for n in range(0, 5):
            time.sleep(1)
            progress = n * 100 / 4
            progress_callback.emit((thread_id, progress))
        return "Done."

    def print_output(self, s):
        print(s)

    def thread_complete(self, thread_id):
        print(f"THREAD #{thread_id} FINALIZADA!")

    def oh_no(self):
        self.thread_id += 1
        worker = Worker(self.execute_this_fn, thread_id=self.thread_id)
        worker.signals.result.connect(self.print_output)
        worker.signals.finished.connect(self.thread_complete)
        worker.signals.progress.connect(self.progress_fn)
        # Inicia a thread nativa do Python
        worker.start()

    def recurring_timer(self):
        self.counter += 1
        self.label.setText(f"Contador: {self.counter}")


# Por causa do notebook, precisamos ver se já existe uma instância do QApplication, caso contrário, criamos uma nova.
if not QApplication.instance():
    app = QApplication([])
else:
    app = QApplication.instance()

window = MainWindow()
app.exec()

Usando threading nativo do Python (threads ativas: 8)
THREAD #1: 0.0% concluída
THREAD #1: 25.0% concluída
THREAD #1: 50.0% concluída
THREAD #2: 0.0% concluída
THREAD #1: 75.0% concluída
THREAD #2: 25.0% concluída
THREAD #1: 100.0% concluída
Done.
THREAD #1 FINALIZADA!
THREAD #2: 50.0% concluída
THREAD #2: 75.0% concluída
THREAD #2: 100.0% concluída
Done.
THREAD #2 FINALIZADA!


0

## Exercícios

### Exercícios Fáceis (básicos de criação e execução)

1. Crie uma classe `Worker` que herde de `QRunnable` e imprima números de 1 a 10 com `time.sleep(0.2)` no método `run()`. Execute-a usando `QThreadPool.globalInstance()`.
2. Modifique o exercício 1 para que o `Worker` receba um parâmetro `max_value` no construtor e conte até esse valor.
3. Crie um `QRunnable` que simule o download de um arquivo (apenas `sleep` + print). Execute 3 instâncias simultâneas no pool.
4. Configure o `QThreadPool` para ter no máximo 4 threads (`setMaxThreadCount`) e execute 10 tarefas simples que apenas imprimem seu ID.
5. Crie um `QRunnable` que calcula a soma dos números de 1 a N e imprime o resultado. Teste com diferentes valores de N.
6. Implemente um `Worker` que dorme por um tempo aleatório entre 0.5 e 2 segundos e depois imprime "Tarefa finalizada".
7. Crie uma aplicação console simples que submete 8 tarefas idênticas ao `QThreadPool` e espera todas terminarem usando `waitForDone()`.
8. Adicione um atributo `task_id` em um `QRunnable` e faça o `run()` imprimir qual tarefa está rodando.
9. Crie um `QRunnable` que lista todos os arquivos de um diretório (usando `os.listdir`) e imprime os nomes.
10. Execute a mesma tarefa de soma em loop 5 vezes usando o pool e observe a paralelização.
11. Configure o `QThreadPool` para usar apenas 2 threads e execute 6 tarefas de sleep de 1 segundo. Observe o comportamento.
12. Crie um `QRunnable` que multiplica dois números passados no construtor e imprime o resultado.
13. Implemente um contador regressivo (de 10 até 0) dentro de um `QRunnable` com delay.
14. Crie uma tarefa que gera 100 números aleatórios e imprime a soma deles.
15. Submeta 20 tarefas muito leves (apenas print) ao pool e verifique quantas threads são realmente criadas.

In [ ]:
# ...


### Exercícios de Dificuldade Média

1. Crie um `QRunnable` que processa uma lista de URLs falsas (strings) simulando requisições com `sleep` e retorna o "status" de cada uma.
2. Implemente um `Worker` que calcula fatorial de um número (recursivo ou iterativo) e imprime o resultado. Execute para vários números.
3. Crie uma classe que herda de `QRunnable` e aceita uma função callback para reportar progresso (usando um atributo ou método).
4. Use `QThreadPool` para redimensionar (simular) 10 imagens (apenas sleep proporcional ao "tamanho"). Limite o pool a 4 threads.
5. Implemente cancelamento básico em um `QRunnable` usando um atributo `self.is_running` verificado no loop.
6. Crie tarefas que processam uma fila (`queue.Queue`) compartilhada entre threads.
7. Faça um `Worker` que busca o maior número primo abaixo de N (simulação com loop e sleep).
8. Execute múltiplas tarefas que compartilham um contador protegido por `QMutex`.
9. Crie um sistema simples de worker pool onde cada tarefa tem prioridade (use `setPriority` do QRunnable).
10. Implemente um `QRunnable` que faz processamento em batch: recebe uma lista, divide em partes e processa cada parte.

In [ ]:
# ...


### Exercícios Difíceis

1. Crie um gerenciador de tarefas reutilizável com `QThreadPool` que aceita funções Python comuns e as envolve automaticamente em `QRunnable`.
2. Implemente um sistema de retry automático (até 3 tentativas) dentro de um `QRunnable` para tarefas que podem falhar.
3. Desenvolva um pipeline de processamento de dados onde várias etapas (`Stage1`, `Stage2`) são executadas em threads diferentes usando o pool e passando dados entre elas.
4. Crie um `QRunnable` que monitora o progresso de um diretório grande (contagem de arquivos recursiva) e atualiza uma variável compartilhada com lock.
5. Implemente controle fino de threads: pause/resume de tarefas individuais usando flags e `QWaitCondition`.

In [ ]:
# ...

## Exercícios Integrados (todos, ou quase todos, os assuntos)

### Exercícios Fáceis

1. Crie uma janela com um botão "Iniciar" que lança 1 `QRunnable` simples. Use `QProgressBar` e um `QLabel` atualizado via sinal.
2. Faça uma interface com `QTextEdit` que recebe logs de várias tarefas rodando em background (usando sinal `pyqtSignal(str)`).
3. Adicione um `QListWidget` que mostra tarefas em andamento. Cada `QRunnable` emite sinal ao terminar.
4. Crie um menu "Arquivo" com ação "Processar" que dispara múltiplas tarefas e mostra `QMessageBox` quando todas finalizarem.
5. Desenvolva uma janela com `QSpinBox` (quantidade de tarefas) e botão para executá-las em paralelo, atualizando um contador de concluídas.
6. Use `QThreadPool` + sinal para preencher um `QTableWidget` com resultados de cálculos (ex: quadrados de números).
7. Crie uma barra de ferramentas com botão de "Cancelar Tudo" que afeta as tarefas em execução (usando flag).
8. Implemente um diálogo (`QDialog`) que roda uma tarefa longa e desabilita o botão OK até terminar.
9. Faça uma aplicação com abas (`QTabWidget`): uma aba para controle de threads e outra para visualização de logs.
10. Adicione um `QSlider` que controla o `maxThreadCount` do pool em tempo real.

In [ ]:
# ...

### Exercícios de Dificuldade Média

1. Crie um gerenciador de downloads falsos com `QProgressBar` por item em um `QScrollArea` (um worker por download).
2. Desenvolva uma aplicação de processamento de imagens (simulado) onde o usuário seleciona arquivos via `QFileDialog` e o pool processa em background, atualizando thumbnails.
3. Implemente um sistema de busca paralela: usuário digita termo, clica em buscar e vários workers vasculham "pastas" virtuais, reportando via sinais.
4. Crie uma janela principal com `QMainWindow`, toolbar e status bar. Use sinais para atualizar a status bar com "X de Y tarefas concluídas".
5. Faça uma janela que abre múltiplas janelas filhas (`QMdiArea`) e cada janela roda seu próprio worker independente.
6. Implemente progresso global + progresso individual usando `QProgressDialog` e sinais personalizados (progresso e mensagem).
7. Crie um editor simples onde salvar um arquivo grande é feito em background. Mostre `QProgressBar` no canto da status bar.

In [ ]:
# ...

### Exercícios Difíceis

1. Desenvolva um "Task Manager" completo com `QTreeView`, onde o usuário pode adicionar, pausar, retomar e cancelar tarefas individualmente. Use sinais customizados e `QThreadPool`.
2. Crie uma aplicação com duas janelas: uma de controle e outra de visualização em tempo real. Use `QTimer` + sinais para atualizar um gráfico (`QChart` ou `matplotlib` embedado) com dados gerados por workers.
3. Implemente um pipeline visual (drag & drop de etapas) onde cada etapa é um worker, e a saída de um alimenta a entrada do próximo usando fila + sinais + `QThreadPool`.

In [ ]:
# ...